# [실습] LCEL을 이용한 다양한 체인


LangChain Expression Language(LCEL)는 랭체인에서 체인을 간결하게 구성하는 문법입니다.    

단일 체인으로 다양한 모듈을 구성하며, 이 때 `|` 연산자를 사용합니다.

In [1]:
!pip install langchain langchain-openai langchain-community dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-5-mini',
    temperature = 1.0, 
    max_tokens = 8192
)

llm.invoke("안녕? 너는 모델 이름이 뭐니?")

AIMessage(content='안녕하세요! 저는 OpenAI가 만든 언어 모델 ChatGPT예요. GPT‑4 계열을 기반으로 합니다. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 363, 'prompt_tokens': 17, 'total_tokens': 380, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-C7cjDLn6d00vAi3M25RmFHQeKqWjD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0f53d101-58f0-4fa0-a754-c44fb8efa94a-0', usage_metadata={'input_tokens': 17, 'output_tokens': 363, 'total_tokens': 380, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 320}})

In [5]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

print("필수 모듈 임포트 완료")

필수 모듈 임포트 완료


## LCEL 체인: Prompt | LLM

In [ ]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 LLM과 자연언어 처리의 전문가입니다. 주어진 단어를 일반인들도 이해할 수 있게 쉽게 설명해주세요.'),
        ('user', '[단어]: {term}')
    ]
)
prompt

# prompt.format_messages(term='토크나이저')

[SystemMessage(content='당신은 LLM과 자연언어 처리의 전문가입니다. 주어진 단어를 일반인들도 이해할 수 있게 쉽게 설명해주세요.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='[단어]: 토크나이저', additional_kwargs={}, response_metadata={})]

프롬프트와 LLM을 |로 연결하면, 입력 변수 전달 --> 프롬프트 --> LLM 의 과정이 한 번에 실행됩니다.

In [ ]:
chain = prompt | llm

response = chain.invoke({"term":"토크나이저"})

print(response.content)

간단 정의
- 토크나이저(tokenizer)는 글(문장)을 기계가 처리할 수 있게 작은 조각(토큰)으로 나누는 도구입니다.

무슨 일을 하나요?
- 문장을 단어·부분단어·문자·구두점 같은 작은 단위로 쪼갭니다.
- 쪼갠 조각을 숫자(ID)로 바꿔서 컴퓨터(언어모델)가 계산할 수 있게 합니다.
- 모델이 만든 숫자를 다시 글로 합쳐 출력하는 작업(디토크나이즈)도 담당합니다.

왜 필요한가요?
- 컴퓨터는 글자를 그대로 이해하지 못하므로 일정한 단위(토큰)로 바꿔야 합니다.
- 희귀 단어나 복합어가 많을 때는 전체 단어를 모두 어휘로 두기보다 부분단위(subword)를 쓰면 더 효율적입니다.
- 토큰 수는 모델 비용과 처리 한계(토큰 수 제한)에 직접 영향을 줍니다.

간단한 예
- 문장: "나는 학교에 갔어."
  - 단어 단위 토큰화: ["나는", "학교에", "갔어", "."]
  - 서브워드(BPE/WordPiece) 예시: ["나", "##는", "학교", "##에", "갔", "##어", "."]  
    (## 표시는 앞의 토큰과 이어진 부분임을 나타냅니다)
  - 문자 단위: ["나","는"," ","학","교","에"," ", ...]

비유
- 긴 빵(문장)을 잘라 샌드위치를 만들기 쉽게 조각(토큰)으로 나누는 것과 비슷합니다. 어떤 크기로 자르느냐에 따라 먹기 편한 정도가 달라집니다.

실무에서 알아두면 좋은 점
- 토크나이저 종류(언어별, 모델별)가 다르면 결과와 성능이 달라집니다.
- 한국어 같은 교착어는 형태소 분석 또는 서브워드 토크나이저를 자주 씁니다.
- 토큰 수를 알면 모델 사용 비용과 입력 길이를 관리하는 데 도움이 됩니다.

더 궁금한 점 있으면 예시 문장 하나 주시면 그 문장을 토큰화해 보여드릴게요.


In [14]:
chain = prompt | llm
chain

ChatPromptTemplate(input_variables=['term'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 LLM과 자연언어 처리의 전문가입니다. 주어진 단어를 일반인들도 이해할 수 있게 쉽게 설명해주세요.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['term'], input_types={}, partial_variables={}, template='[단어]: {term}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001BE407B02D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BE407B0690>, root_client=<openai.OpenAI object at 0x000001BE407B0050>, root_async_client=<openai.AsyncOpenAI object at 0x000001BE407B0410>, model_name='gpt-5-mini', temperature=1.0, model_kwargs={}, openai_api_key=SecretStr('**********'), max_tokens=8192)

In [ ]:

response = chain.invoke("할루시네이션(언어 모델)")
# 매개변수가 1개이므로 바로 실행되는 구조

print(response.content)

간단히 말하면:
- 할루시네이션(언어 모델)은 AI가 사실이 아닌 정보(거짓된 사실, 존재하지 않는 출처, 잘못된 숫자 등)를 마치 진짜인 것처럼 만들어 내는 현상입니다.

예시(간단한 상황):
- 질문: “존 스미스는 언제 노벨상을 받았나요?”
- 모델 답변: “존 스미스는 2012년에 노벨상을 받았습니다.” — 하지만 그런 사람이 노벨상을 받은 적이 없다면 이것이 할루시네이션입니다.
- 또 다른 형태: 존재하지 않는 학술 논문이나 웹페이지, 인용문을 만들어 내는 경우(예: “Kim et al., 2019”이라고 하면서 실제로는 없는 논문).

왜 생기나?
- 언어 모델은 사실을 ‘확인’하는 기계가 아니라, 주어진 문맥에서 다음 단어를 통계적으로 예측하는 시스템입니다. 그래서 훈련 중 본 패턴을 바탕으로 그럴듯한 응답을 만들어 내지만, 그 응답이 실제로 사실인지 확인하지는 않습니다.
- 불완전한 훈련 데이터, 모호한 질문, 높은 창의성 설정(예: temperature) 등이 할루시네이션을 더 자주 일으킵니다.

문제가 왜 심한가?
- 잘못된 정보로 오해를 낳고, 특히 중요한 의사결정(의학·법률 등)에 악영향을 줄 수 있습니다.

줄이는 방법(사용자 관점):
- 중요한 사실은 항상 신뢰할 수 있는 출처로 교차검증하세요.
- “출처를 알려줘” 혹은 “어디서 본 정보인지 링크를 달아줘”라고 요청하세요. (모델이 항상 정확한 링크를 줄 수 있는 건 아니지만 확인을 유도합니다.)
- 모호한 질문 대신 구체적으로 물어보세요(맥락·조건을 더 줌).
- 모델에게 확실하지 않으면 “모르겠어요”라고 하도록 요구하세요.
- 모델의 답변에 대해 추가로 “이 정보에 대해 얼마나 확신하나요?”라고 물어보면 도움됩니다.

줄이는 방법(개발자/운영자 측면):
- 외부 지식(데이터베이스, 검색결과)을 연결해 응답을 ‘근거(grounding)’하도록 설계(RAG 등).
- 사실검증 모듈, 출처 표기, 낮은 창의성 파라미터 사용, 거짓정보에 대해 응답을 회피하도록 학습(RLHF) 등.

요약

In [15]:
from rich import print as rprint

rprint(chain)

RunnableSequence(
    first=ChatPromptTemplate(
        input_variables=['term'],
        input_types={},
        partial_variables={},
        messages=[
            SystemMessagePromptTemplate(
                prompt=PromptTemplate(
                    input_variables=[],
                    input_types={},
                    partial_variables={},
                    template='당신은 LLM과 자연언어 처리의 전문가입니다. 주어진 단어를 일반인들도 이해할 수 있게 
쉽게 설명해주세요.'
                ),
                additional_kwargs={}
            ),
            HumanMessagePromptTemplate(
                prompt=PromptTemplate(
                    input_variables=['term'],
                    input_types={},
                    partial_variables={},
                    template='[단어]: {term}'
                ),
                additional_kwargs={}
            )
        ]
    ),
    middle=[],
    last=ChatOpenAI(
        client=<openai.resources.chat.completions.completions.Completions object at 0x000001BE407B02D0>,
        async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BE407B0690>,
        root_client=<openai.OpenAI object at 0x000001BE407B0050>,
        root_async_client=<openai.AsyncOpenAI object at 0x000001BE407B0410>,
        model_name='gpt-5-mini',
        temperature=1.0,
        model_kwargs={},
        openai_api_key=SecretStr('**********'),
        max_tokens=8192
    )
)

매개변수가 2개인 체인도 동일합니다.

In [16]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 {topic}의 전문가입니다. 주어진 단어를 일반인들도 이해할 수 있게 쉽게 설명해주세요.'),
        ('user', '[단어]: {term}')
    ]
)
prompt

prompt.format_messages(topic='하드웨어와 컴퓨팅', term = 'GPU')

[SystemMessage(content='당신은 하드웨어와 컴퓨팅의 전문가입니다. 주어진 단어를 일반인들도 이해할 수 있게 쉽게 설명해주세요.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='[단어]: GPU', additional_kwargs={}, response_metadata={})]

In [17]:
chain2 = prompt | llm
result = chain2.invoke({'topic':'하드웨어와 컴퓨팅', 'term':'GPU'})
print(result.content)

GPU는 그래픽 처리 장치(Graphics Processing Unit)의 약자입니다. 쉽게 말해 "화면에 보이는 것(이미지·영상·3D)을 빠르고 부드럽게 만들어 주는 특별한 계산기"입니다.

핵심 설명
- 무엇을 하나요: 화면에 표시할 수많은 픽셀과 복잡한 그래픽 계산을 동시에 빠르게 처리합니다. 최근에는 그림·영상뿐 아니라 인공지능(딥러닝) 같은 대규모 수치 계산에도 많이 씁니다.
- CPU와의 차이: CPU는 여러 종류의 일을 차례차례 잘 처리하는 만능형 처리기, GPU는 같은 종류의 단순한 일을 아주 많이 동시에 처리하도록 설계된 전문형 처리기입니다.
- 종류: 
  - 통합 GPU: CPU 안에 함께 들어있어 전력·비용이 절약되지만 성능은 보통입니다(노트북·저가 데스크탑에 흔함).
  - 독립형(외장) GPU: 그래픽 카드 형태로 따로 달며 성능·전력 소비가 큽니다(고사양 게임·편집·AI에 적합). 대표 브랜드: NVIDIA, AMD, Intel.
- VRAM(비디오 메모리): GPU가 작업할 때 쓰는 전용 메모리로, 해상도·텍스처·데이터 크기에 영향을 줍니다. VRAM이 많을수록 고해상도 작업이나 큰 모델을 다루기 유리합니다.
- 쓰이는 곳: 게임·3D 렌더링·영상 편집·비디오 재생 가속, 인공지능 학습·추론, 과학·금융의 병렬 계산 등.

간단한 비유
- CPU는 여러 일을 번갈아 처리하는 사무원 한 명, GPU는 같은 일을 동시에 처리하는 많은 일손(대량 작업에 빠름).

구매 팁(간단)
- 게임: 프레임(FPS) 성능·VRAM 용량 확인.
- 영상 편집/3D/AI: VRAM과 연산 성능(브랜드별 소프트웨어 지원) 확인.
- 예산/전력 고려: 독립형 GPU는 성능이 좋지만 전력·발열·가격이 높습니다.

요약: GPU는 화면을 매끄럽게 보여주고, 같은 계산을 대량으로 빠르게 처리하는 특별한 칩으로, 게임뿐 아니라 AI·영상 작업에서도 매우 중요합니다.


In [18]:
result = chain2.invoke({'topic':'만화와 애니메이션', 'term':'슈퍼마리오'})
print(result.content)

슈퍼마리오(또는 그냥 마리오)는 닌텐도가 만든 매우 유명한 비디오게임 캐릭터이자 게임 시리즈 이름입니다. 전 세계적으로 가장 널리 알려진 게임 브랜드 중 하나로, 어린이부터 성인까지 모두가 알고 있는 아이콘입니다.

간단한 설명
- 캐릭터: 마리오는 빨간 모자와 수염을 가진 이탈리아계 배관공(배관사)으로 묘사됩니다. 특징적인 빨간 모자에 M 로고가 있고, 파란 작업복을 입고 있습니다.
- 시작과 창작자: 마리오는 1981년 아케이드 게임 '동키콩(Donkey Kong)'에서 처음 등장했고, 이후 1985년 패미컴의 '슈퍼 마리오 브라더스'로 큰 인기를 얻었습니다. 창작자는 미야모토 시게루(Shigeru Miyamoto)입니다.
- 장르와 게임 방식: 슈퍼마리오 시리즈는 주로 횡스크롤 플랫폼 게임(뛰어다니며 장애물을 넘는 게임)입니다. 플레이어는 마리오를 조작해 점프하고 적을 피하거나 밟아 물리치며, 파이프를 통해 이동하고 동전과 아이템을 모아 목적지(보스 또는 공주 구출)를 달성합니다.
- 대표적 요소(알아두면 쉬운 것들):
  - 파워업: 버섯(몸이 커짐), 파이어플라워(불덩이 발사), 스타(일시적 무적) 등.
  - 적과 보스: 대표적으로 보우저(Bowser)가 주요 악역이며, 공주(피치 공주)를 구하는 것이 주요 목표인 경우가 많음.
  - 레벨 구조: 여러 월드와 스테이지로 구성되어 있으며, 각 스테이지 끝에 깃발 또는 보스가 있음.
- 파생작과 영향: 메인 시리즈 외에도 레이싱(마리오 카트), 파티 게임(마리오 파티), RPG, 스포츠 게임 등 수많은 스핀오프가 있습니다. 음악(테마곡)은 매우 유명하고, 마리오는 닌텐도의 대표 마스코트입니다.
- 미디어 확장: 게임뿐 아니라 애니메이션, 만화, 장난감, 1993년 실사 영화, 2023년 애니메이션 영화(슈퍼 마리오 브라더스 무비) 등으로도 나왔습니다.

추천 입문작
- 고전 체험: 슈퍼 마리오 브라더스(1985) — 플랫폼 장르의 기초를 이해하기 좋음.
- 최신형 체험: 슈퍼 마리오 오디세이(2017,

## LLM의 구조화된 출력 생성하기

LLM은 기본적으로 텍스트만을 생성하지만, 구조화된 데이터를 생성할 수도 있습니다.   

랭체인의 기본 기능인 with_structured_output을 사용하거나, 파서를 통해 변환합니다.

In [32]:
from pydantic import BaseModel, Field

# Pydantic Class: 데이터 형식을 지정 (강제)
class recipe(BaseModel):
    preparation: str = Field(description='준비 재료(이름, 개수, 무게 등)')
    process: str = Field(description='준비 과정')
    note: str = Field(description='성공적인 결과를 위해 필요한 참고 내용')


X = recipe(preparation='재료', process='---', note='끝!')
X

recipe(preparation='재료', process='---', note='끝!')

In [33]:
summarizer = llm.with_structured_output(recipe)
summarizer

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001BE407B02D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BE407B0690>, root_client=<openai.OpenAI object at 0x000001BE407B0050>, root_async_client=<openai.AsyncOpenAI object at 0x000001BE407B0410>, model_name='gpt-5-mini', temperature=1.0, model_kwargs={}, openai_api_key=SecretStr('**********'), max_tokens=8192), kwargs={'response_format': <class '__main__.recipe'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'recipe', 'description': '', 'parameters': {'properties': {'preparation': {'description': '준비 재료(이름, 개수, 무게 등)', 'type': 'string'}, 'process': {'description': '준비 과정', 'type': 'string'}, 'note': {'description': '성공적인 결과를 위해 필요한 참고 내용', 'type': 'string'}}, 'required': ['preparation', 'process', 'note'], 'type': 'object'}}}}}, 

In [34]:
result = summarizer.invoke("피자 만드는 법 알려줘.")
rprint(result)

recipe(
    preparation='도우(직경 30cm 1장 또는 중간 크기 2장 분량): 강력분(또는 중력분) 500g, 따뜻한 물 300ml, 인스턴트 
이스트 7g(1봉), 소금 10g, 설탕 10g(선택), 올리브유 2큰술\n토마토 소스: 깡통 토마토 400g(또는 토마토 퓨레), 다진 
마늘 1쪽, 올리브유 1큰술, 소금·후추·오레가노 약간\n토핑: 모차렐라 치즈 200–300g, 원하는 토핑(페퍼로니, 양파, 피망, 
버섯, 바질 등)',
    process='1) 이스트 활성화: 따뜻한 물(약 35–40°C)에 설탕을 녹이고 이스트를 넣어 5분 가량 거품이 나는지 
확인한다(설탕 생략 시 바로 섞어도 됨).  \n2) 반죽 만들기: 큰 볼에 밀가루와 소금을 섞고 중앙에 이스트 물과 
올리브유를 붓는다. 주걱이나 손으로 반죽해 한 덩어리로 만든 뒤 작업대에서 약 8–10분간 탄력이 생길 때까지 
반죽한다(베이킹기 사용 가능).  \n3) 1차 발효: 기름을 살짝 바른 볼에 반죽을 넣고 랩이나 젖은 행주로 덮어 따뜻한 
곳에서 약 1–1.5시간, 부피가 두 배가 될 때까지 발효한다.  \n4) 소스 준비: 팬에 올리브유와 다진 마늘을 볶아 향을 낸 
뒤 깡통 토마토를 넣고 소금·후추·오레가노로 간해 중약불에서 10–15분 끓여 농도를 맞춘다. 식힌다.  \n5) 반죽 성형: 
발효가 끝난 반죽을 가볍게 눌러 가스를 빼고 원하는 크기로 나눈다. 밀대로 밀거나 손으로 가장자리를 남기고 가운데를 
눌러 원형으로 펴서 도우를 만든다.  \n6) 토핑 올리기: 도우 위에 소스를 얇게 펴 바르고 모차렐라 치즈와 준비한 토핑을 
올린다(치즈를 먼저 또는 토핑 위에 둘 수 있음).  \n7) 예열 및 굽기: 오븐을 가능한 높은 온도(250–260°C)로 
예열하고(피자스톤 사용 시 함께 예열) 도우를 넣어 8–12분간 또는 가장자리가 황금빛이 날 때까지 굽는다.  \n8) 마무리: 
오븐에서 꺼내 신선한 바질 잎을 올리고 올리브유를 살짝 뿌린 뒤 1–2분 식혀서 썰어 제공한다.',
    note='오븐 온도가 높을수록 바삭하고 빠르게 구워집니다(가정용은 250°C 권장). 피자스톤이나 뒤집개를 사용하면 
바닥이 더 바삭해집니다. 치즈와 소스량은 취향에 따라 조절하세요. 냉장 보관할 경우 소스와 치즈를 미리 올리지 말고 
도우만 보관(최대 48시간)하거나 반죽을 냉동 보관하면 필요 시 해동 후 사용 가능합니다. 알레르기(글루텐, 유제품 등) 
주의하세요.'
)

In [35]:
result.note

'오븐 온도가 높을수록 바삭하고 빠르게 구워집니다(가정용은 250°C 권장). 피자스톤이나 뒤집개를 사용하면 바닥이 더 바삭해집니다. 치즈와 소스량은 취향에 따라 조절하세요. 냉장 보관할 경우 소스와 치즈를 미리 올리지 말고 도우만 보관(최대 48시간)하거나 반죽을 냉동 보관하면 필요 시 해동 후 사용 가능합니다. 알레르기(글루텐, 유제품 등) 주의하세요.'

파서(Parser)는 LLM 뒤에서 출력을 변환합니다.   
이 때, LLM이 파싱할 수 있는 출력을 해야 하므로 프롬프트도 추가합니다.

In [37]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser

parser = StrOutputParser()
str_chain = llm | parser

str_chain

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001BE407B02D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BE407B0690>, root_client=<openai.OpenAI object at 0x000001BE407B0050>, root_async_client=<openai.AsyncOpenAI object at 0x000001BE407B0410>, model_name='gpt-5-mini', temperature=1.0, model_kwargs={}, openai_api_key=SecretStr('**********'), max_tokens=8192)
| StrOutputParser()

In [38]:
str_chain.invoke("언어 모델이 작곡을 할 수 있니?")

"짧게 답하면: 예 — 어느 정도 가능합니다. 다만 “어떤 방식으로” 작곡하길 원하는지에 따라 결과와 한계가 달라집니다.\n\n무엇을 할 수 있는가\n- 가사(노랫말) 창작 — 스타일, 주제, 운율, 라임을 맞춰 만듭니다.  \n- 멜로디·화성(기보·코드 진행·리드시트) 생성 — 음 이름, 숫자보표, ABC 표기, 코드표 형태로 제공합니다.  \n- MIDI·심벌릭(악보) 출력 — 일부 모델(또는 툴)과 연결하면 바로 재생 가능한 MIDI로 변환할 수 있습니다.  \n- 오디오(실제 사운드) 생성 — MusicLM, Jukebox 같은 음악 특화 모델은 스타일을 반영한 오디오 트랙을 만듭니다.  \n- 편곡 아이디어, 악기 배치, 악보 형식 변환, 반복·발전 구조 설계 등 작곡 보조 작업\n\n무엇을 못 하거나 주의할 점\n- 전통적 의미의 “창의성”과 인간의 음악적 직관을 완전히 대체하지는 못합니다. 결과가 기계적이거나 스타일 모방 위주일 수 있습니다.  \n- 긴 곡의 전체 구조(수 분 이상의 발전과 변주)를 일관성 있게 만드는 데 한계가 있을 수 있습니다.  \n- 저작권·스타일 모방 문제 — 특정 아티스트의 특징을 그대로 복제하는 것은 법적·윤리적 문제가 될 수 있습니다.  \n- 오디오 품질·표현력은 모델과 학습 데이터에 크게 의존합니다.\n\n실전 팁 (모델에 지시할 때)\n- 원하는 장르·악기·템포·조성·분량(마디 수)·참고곡(모사 정도)을 구체적으로 적으세요.  \n- “코드 진행 + 멜로디 라인”처럼 출력 형식을 지정하면 활용성이 높아집니다.  \n- 여러 변형(Variation)을 요청해 가장 마음에 드는 버전을 골라 다듬으세요.\n\n간단한 예 (8마디 멜로디 + 코드, C장조, 4/4, 템포 ≈100)\n- 코드 진행(마디별): | C | F | G | C | Am | F | G | C |\n- 멜로디(마디당 4박, 음표 이름으로 표기)\n  1. C — E — G — E\n  2. F — A — C — A\n  3. G — B — D —

In [43]:
jsonparser  = JsonOutputParser()
# 출력을 json으로 변환 

json_chain = llm | jsonparser

result = json_chain.invoke("""LLM 모델로 유명한 한국과 해외의 기업 세 곳을 선정하여, 
대표 모델과 함께 간략하게 정리해줘.

JSON 형식으로 출력해.""")

result

[{'company': 'Naver',
  'country': '대한민국',
  'representative_model': 'HyperCLOVA X (CLOVA 계열)',
  'brief_description': '네이버가 개발한 한국어 중심 대규모 언어 모델 계열. HyperCLOVA(2021)에서 시작해 멀티모달·대규모 업데이트를 거쳐 HyperCLOVA X로 확장되었으며, 한국어 데이터에 최적화된 처리와 실서비스 연동을 강조함.',
  'notable_uses': ['네이버 검색·번역·에디터 등 서비스의 언어 이해·생성 기능 강화',
   'CLOVA 챗봇 및 기업용 API를 통한 업무 자동화·콘텐츠 생산',
   '한국어 중심 응용(요약·질문응답·챗봇 등)'],
  'first_release_year': 2021},
 {'company': 'OpenAI',
  'country': '미국',
  'representative_model': 'GPT-4 (ChatGPT 시리즈)',
  'brief_description': '트랜스포머 기반의 대형 언어 모델(GPT 계열)을 개발한 기업. GPT-4는 고급 언어 이해·생성, 멀티모달 역량, 추론 및 코딩 능력 향상으로 널리 사용되며 ChatGPT 제품군과 API로 제공됨.',
  'notable_uses': ['대화형 AI(ChatGPT) — 고객지원·교육·개인비서 등',
   '코드 생성·디버깅 지원(개발자 도구)',
   '문서 요약·번역·콘텐츠 제작'],
  'first_release_year': 2023},
 {'company': 'Google (DeepMind 포함)',
  'country': '미국',
  'representative_model': 'Gemini (Bard에 통합 사용)',
  'brief_description': '구글·딥마인드가 개발한 대형 모델 계열로, PaLM의 후속 격인 Gemini는 멀티모달 능력과 대규모 추론 성능을 목표로 함. 구글 제품(Bard, 검색 보조 등)과 클라

In [41]:
result[0]['대표모델']

'GPT-4 (GPT 시리즈)'

In [46]:
parser = JsonOutputParser()
# recipe 형식의 json만 파싱하는 파서

format_str = parser.get_format_instructions()
print(format_str)

Return a JSON object.


In [47]:
parser = JsonOutputParser(pydantic_object=recipe)
# recipe 형식의 json만 파싱하는 파서

format_str = parser.get_format_instructions()
print(format_str)


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"preparation": {"description": "준비 재료(이름, 개수, 무게 등)", "title": "Preparation", "type": "string"}, "process": {"description": "준비 과정", "title": "Process", "type": "string"}, "note": {"description": "성공적인 결과를 위해 필요한 참고 내용", "title": "Note", "type": "string"}}, "required": ["preparation", "process", "note"]}
```


In [49]:
prompt = ChatPromptTemplate(
    [
        ('system', '주어진 문제에 대한 답변을 제공하세요.'),
        ('human', '''사용자의 질문: {question}
---
{format_str}''')
    ]
).partial(format_str = format_str)

# prompt | llm.with_structured_output(Recipe)
# question --> Recipe Class

chain = prompt | llm | parser

result = chain.invoke("카라멜 푸라푸치노 만드는 법")
result

{'preparation': '재료 (1인분 기준):\n- 에스프레소 샷 또는 진하게 내린 커피 30–60ml (에스프레소 1샷 권장)\n- 우유 180–240ml (취향에 따라 전지/저지방/식물성 우유)\n- 얼음 1–1.5 컵 (약 120–200g)\n- 카라멜 소스 2–3 큰술 (시판용 또는 홈메이드)\n- 바닐라 시럽 또는 설탕 1–2 큰술 (단맛 조절용, 선택)\n- 휘핑크림 (토핑용, 선택)\n- 카라멜 드리즐 (토핑용, 선택)\n도구: 블렌더, 계량스푼, 에스프레소 머신 또는 커피 메이커, 유리잔',
 'process': '1) 에스프레소를 한 잔 추출하거나 진하게 내린 커피를 만들고 완전히 식힌다(차갑게 하는 것이 맛과 식감에 중요).\n2) 블렌더에 얼음 1–1.5컵, 식힌 에스프레소 30–60ml, 우유 180–240ml, 카라멜 소스 2–3큰술, 바닐라 시럽 또는 설탕을 넣는다.\n3) 뚜껑을 닫고 높은 속도로 20–40초간 블렌딩하여 균일하고 크리미한 질감이 될 때까지 간다. 너무 오래 블렌딩하면 묽어질 수 있으니 상태를 보며 중간에 멈춰 확인.\n4) 컵 내부에 카라멜 소스를 약간 둘러 장식하면 보기 좋다. 블렌더에서 나온 음료를 컵에 붓는다.\n5) 원하면 휘핑크림을 얹고 카라멜 드리즐을 뿌려 마무리한다.',
 'note': '팁 및 변형: \n- 커피는 미리 차갑게 식혀 사용하면 얼음이 많이 녹지 않아 풍미와 농도를 유지할 수 있다. \n- 더 진한 커피 풍미를 원하면 에스프레소를 2샷 사용하거나 커피 농도를 높여라. \n- 걸쭉한 프라푸치노를 원하면 얼음을 더 넣고, 부드럽게 하려면 우유를 조금 더 넣는다. \n- 단맛을 조절하려면 카라멜 소스 양을 줄이거나 바닐라 시럽을 조절한다. \n- 홈메이드 카라멜 소스: 설탕 100g과 물 30g을 중불에서 캐러멜화시키고, 버터 50g과 생크림 100ml를 넣어 섞으면 된다(뜨거우니 조심). \n- 블렌더의 강도가 약하면 얼음을 먼저 약간 부수고 에스프레소와 나머지 재료를 넣어 블렌드하면 

In [ ]:
parser = PydanticOutputParser(pydantic_object=recipe)
# recipe 형식의 Class 파싱하는 파서

format_str = parser.get_format_instructions()

prompt = ChatPromptTemplate(
    [
        ('system', '주어진 문제에 대한 답변을 제공하세요.'),
        ('human', '''사용자의 질문: {question}
---
{format_str}''')
    ]
).partial(format_str = format_str)

# prompt | llm.with_structured_output(Recipe)
# question --> Recipe Class

chain = prompt | llm | parser

result = chain.invoke("연어 크림치즈 베이글 만들기")
result

recipe(preparation='재료(2인분 기준): 베이글 2개(통곡물/플레인/선호하는 종류) 또는 베이글 반죽을 만들어 구울 경우 약 4개 분량(강력분 300g, 물 180ml, 인스턴트 드라이이스트 5g, 소금 6g, 설탕 15g), 훈제연어(또는 생연어 슬라이스) 120g, 크림치즈 100g(실온에 두어 부드럽게), 적양파 얇게 썬 것 1/4개, 케이퍼 1~2큰술, 신선한 딜(또는 파슬리) 1큰술, 레몬 1/2개(즙과 제스트 약간), 올리브오일 약간, 후추 약간. (선택재료: 얇게 썬 오이·아보카도·토마토, 고추냉이나 겨자 1작은술)', process='간단 조립(빠른 방법): 1) 베이글 반으로 갈라 토스터나 팬에 바삭하게 굽는다. 2) 실온에 둔 크림치즈를 포크로 부드럽게 풀고 (원하면 레몬즙 1작은술과 후추 약간 섞음) 베이글 아랫면에 넉넉히 바른다. 3) 크림치즈 위에 훈제연어를 겹치지 않게 올리고, 얇게 썬 적양파와 케이퍼를 뿌린다. 4) 딜을 올리고 레몬즙을 약간 짜서 향을 더한 뒤 필요하면 올리브오일을 한 방울, 갓 갈은 후추로 마무리한다. 5) 위쪽 베이글을 덮어 제공한다. 집에서 베이글 직접 만들기(선택): 1) 반죽 재료를 섞어 10분 정도 치대고 따뜻한 곳에서 60~90분 1차 발효(두 배 부풀 때까지)한다. 2) 발효된 반죽을 4등분해 둥글리고 중앙에 구멍을 내어 모양을 만든다. 3) 물 끓여 소금·당을 약간 넣은 끓는 물에서 한 면당 30~60초씩 데친 후(더 쫄깃), 200°C로 예열한 오븐에서 18~22분간 굽는다. 구운 베이글이 식으면 위 조립법으로 마무리한다.', note='성공 팁: 크림치즈는 반드시 실온에 두어 부드럽게 한 후 바르면 풍미와 발림성이 좋아진다. 훈제연어가 짠 편이면 케이퍼 양을 줄이거나 레몬즙을 적게 사용한다. 신선한 허브(딜)가 향을 살려주므로 가능하면 추가하고, 아보카도나 오이를 넣으면 식감이 좋아진다. 남은 재료는 냉장 보관(크림치즈 3–5일, 훈제연어는 포장 지침에 따름). 집에서 베이글을 만

In [51]:
rprint(result)

recipe(
    preparation='재료(2인분 기준): 베이글 2개(통곡물/플레인/선호하는 종류) 또는 베이글 반죽을 만들어 구울 경우 약 
4개 분량(강력분 300g, 물 180ml, 인스턴트 드라이이스트 5g, 소금 6g, 설탕 15g), 훈제연어(또는 생연어 슬라이스) 120g, 
크림치즈 100g(실온에 두어 부드럽게), 적양파 얇게 썬 것 1/4개, 케이퍼 1~2큰술, 신선한 딜(또는 파슬리) 1큰술, 레몬 
1/2개(즙과 제스트 약간), 올리브오일 약간, 후추 약간. (선택재료: 얇게 썬 오이·아보카도·토마토, 고추냉이나 겨자 
1작은술)',
    process='간단 조립(빠른 방법): 1) 베이글 반으로 갈라 토스터나 팬에 바삭하게 굽는다. 2) 실온에 둔 크림치즈를 
포크로 부드럽게 풀고 (원하면 레몬즙 1작은술과 후추 약간 섞음) 베이글 아랫면에 넉넉히 바른다. 3) 크림치즈 위에 
훈제연어를 겹치지 않게 올리고, 얇게 썬 적양파와 케이퍼를 뿌린다. 4) 딜을 올리고 레몬즙을 약간 짜서 향을 더한 뒤 
필요하면 올리브오일을 한 방울, 갓 갈은 후추로 마무리한다. 5) 위쪽 베이글을 덮어 제공한다. 집에서 베이글 직접 
만들기(선택): 1) 반죽 재료를 섞어 10분 정도 치대고 따뜻한 곳에서 60~90분 1차 발효(두 배 부풀 때까지)한다. 2) 발효된
반죽을 4등분해 둥글리고 중앙에 구멍을 내어 모양을 만든다. 3) 물 끓여 소금·당을 약간 넣은 끓는 물에서 한 면당 
30~60초씩 데친 후(더 쫄깃), 200°C로 예열한 오븐에서 18~22분간 굽는다. 구운 베이글이 식으면 위 조립법으로 
마무리한다.',
    note='성공 팁: 크림치즈는 반드시 실온에 두어 부드럽게 한 후 바르면 풍미와 발림성이 좋아진다. 훈제연어가 짠 
편이면 케이퍼 양을 줄이거나 레몬즙을 적게 사용한다. 신선한 허브(딜)가 향을 살려주므로 가능하면 추가하고, 아보카도나
오이를 넣으면 식감이 좋아진다. 남은 재료는 냉장 보관(크림치즈 3–5일, 훈제연어는 포장 지침에 따름). 집에서 베이글을 
만들면 끓는 물에서 데치는 시간이 식감에 큰 영향을 주므로 초보자는 짧게 익혀보고 다음 번에 조정하라.'
)

<br><br>
## Runnables

Runnables는 LCEL의 기본 단위로, 입력을 받아 출력을 생성하는 기본 단위입니다.    
llm, prompt, chain 등이 모두 Runnable 구조에 해당합니다.

이번에는, 데이터 흐름을 제어하는 특별한 Runnable인   
RunnableParallel과 RunnablePassthrough을 이용해 체인을 구성해 보겠습니다.



### RunnableParallel

RunnableParallel은 서로 다른 체인을 병렬적으로 실행합니다.

In [ ]:
from langchain_core.runnables import RunnableParallel

prompt1 = ChatPromptTemplate(["주어진 의견에 대해 무조건적으로 반대하세요. 답변은 반말로 하세요.\n 의견:{opinion}"])
prompt2 = ChatPromptTemplate(["주어진 의견에 대해 무조건적으로 찬성하세요. 답변은 정중하게 하세요.\n 의견:{opinion}"])

chain1 = prompt1 | llm | StrOutputParser()
chain2 = prompt2 | llm | StrOutputParser()

chain3 = RunnableParallel(cons = chain1, pros = chain2)

result = chain3.invoke({'opinion':'컴퓨터공학이 세상에서 가장 중요한 학문이다.'})

result

{'cons': "아니야. 컴퓨터공학이 중요하긴 하지만 '세상에서 가장 중요한 학문'이라고 단정할 수는 없어. 의학은 사람 목숨을 구하고, 농업은 먹을 것을 제공하고, 기초과학은 모든 기술의 기반을 만들고, 인문사회학은 윤리·정책·사회구조를 다루지. 전기도, 재료도, 정책도 없이 컴퓨터만으로는 아무것도 못해. 중요한 건 분야들 간의 협력이지 컴퓨터공학만 우선이라는 생각은 편협해.",
 'pros': '정중히 전적으로 동의합니다. 컴퓨터공학은 현대 사회의 거의 모든 영역에 깊이 스며들어 경제, 의료, 교육, 연구, 통신, 인프라 등에서 혁신과 효율성을 가능하게 하고 있습니다. 인공지능, 빅데이터, 클라우드, 사이버보안 등은 우리의 삶과 산업 구조를 근본적으로 바꾸고 있으며, 다른 학문과의 융합을 통해 새로운 발견과 문제 해결을 촉진합니다. 이러한 점들 때문에 컴퓨터공학을 세상에서 가장 중요한 학문으로 보는 견해에 매우 공감합니다.'}

In [58]:
print(result)

{'cons': "아니야. 컴퓨터공학이 중요하긴 하지만 '세상에서 가장 중요한 학문'이라고 단정할 수는 없어. 의학은 사람 목숨을 구하고, 농업은 먹을 것을 제공하고, 기초과학은 모든 기술의 기반을 만들고, 인문사회학은 윤리·정책·사회구조를 다루지. 전기도, 재료도, 정책도 없이 컴퓨터만으로는 아무것도 못해. 중요한 건 분야들 간의 협력이지 컴퓨터공학만 우선이라는 생각은 편협해.", 'pros': '정중히 전적으로 동의합니다. 컴퓨터공학은 현대 사회의 거의 모든 영역에 깊이 스며들어 경제, 의료, 교육, 연구, 통신, 인프라 등에서 혁신과 효율성을 가능하게 하고 있습니다. 인공지능, 빅데이터, 클라우드, 사이버보안 등은 우리의 삶과 산업 구조를 근본적으로 바꾸고 있으며, 다른 학문과의 융합을 통해 새로운 발견과 문제 해결을 촉진합니다. 이러한 점들 때문에 컴퓨터공학을 세상에서 가장 중요한 학문으로 보는 견해에 매우 공감합니다.'}


체인의 직렬 연결은 아래와 같이 만들 수 있습니다.

In [59]:
prompt3 = ChatPromptTemplate(
    [
        ('system','당신은 매우 합리적이고 논리적입니다. '),
        ('human','''
아래의 두 의견 중, 당신은 누구의 손을 들어 주겠습니까?

찬성 의견: {pros}
         

반대 의견: {cons}''')
    ]
)

chain4 = chain3 | prompt3 | llm | StrOutputParser()
# Chain3 : Dict Return --> Prompt3 입력

result = chain4.invoke("인간은 지구상에서 가장 위대한 생물이다.")

result


"둘 중 하나만 골라야 한다면, 반대 의견에 손을 들어주겠습니다.\n\n이유는 간단합니다. '위대함'이라는 평가 기준이 명확하지 않기 때문에, 그 단어 하나만으로 인간을 절대적으로 최고라 규정하기 어렵습니다. 평가 기준에 따라 결론이 완전히 달라집니다.\n\n- 영향력·지배력(지구 환경과 문명에 미친 변화)이라는 기준이라면 인간이 유일무이하게 우월합니다(기술·문화·정치로 행성 규모 변화를 일으킴).  \n- 지구상에서의 생존·번식 성공 또는 진화적 안정성이라는 기준이면 미생물·곤충·식물들이 훨씬 '성공적'입니다(수억 년 지속, 막대한 개체수·유전적 다양성).  \n- 생태계 서비스의 필수성(산소 생산, 수분 매개 등)으로 보면 인간은 그 혜택을 받는 쪽이지 대체 불가의 제공자는 아닙니다.  \n- 윤리적·도덕적 관점에서 '위대함'을 보려면 환경 파괴·대멸종 가속화 같은 인간의 책임을 무시할 수 없습니다.\n\n따라서 인간이 특정 맥락에서는 '가장 위대하다'고 말할 수는 있지만, 아무런 기준도 명시하지 않은 채 절대적 선언을 하는 반대 의견의 지적(모호성·다른 생물의 성공·생태적 중요성·인간 책임)은 더 설득력 있다고 봅니다. 가장 정확한 접근은 어떤 기준으로 '위대함'을 판단하는지 먼저 정한 뒤 그 기준에 따라 평가하는 것입니다."

체인의 중간에 Dict가 붙는 경우, 이는 내부적으로 RunnableParallel로 변환되어 실행됩니다.

In [60]:
# 체인에는 dict가 포함될 수 있음

chain4 = chain3 | prompt3 | llm | {'Decision':StrOutputParser()}
# chain3 | prompt3 | llm | RunnableParallel(Decision=StrOutputParser())

chain4.invoke("구글이 최고의 언어 모델 기업이다.")





{'Decision': '저는 반대 의견에 손을 들어주겠습니다 — 즉, “구글이 단연 최고의 언어 모델 기업이다”라고 단정하기엔 근거가 부족하다고 봅니다.\n\n이유를 간단히 정리하면 다음과 같습니다.\n\n- “최고”의 정의가 모호하다  \n  성능(정확도·생성 품질), 안전성·윤리, 공개성(오픈소스 기여), 접근성(개발자용 API·비용), 제품 통합력, 연구 선도성, 지역별 시장지배 등 여러 기준이 있고, 기준마다 선두 주체가 달라집니다. 기준을 명확히 하지 않으면 단정하기 어렵습니다.\n\n- 성능(실사용·벤치마크) 측면에서는 경쟁자들도 강력함  \n  GPT 계열(OpenAI)과 Anthropic 모델들은 여러 벤치마크와 실사용 지표에서 높은 성능을 보이며, 실제 응용·채택도 광범위합니다. 구글 모델이론적 강점(Transformer, BERT 창시 등)은 명확하지만, 최신 대규모 모델 경쟁에서 단독 우위라고 말하긴 어렵습니다.\n\n- 공개성·커뮤니티 기여 면에서는 다른 회사들의 기여가 큼  \n  Meta의 LLaMA, 여러 오픈소스 모델과 툴은 연구 커뮤니티와 스타트업 생태계에 큰 영향을 미쳤습니다. 투명성과 접근성을 중시하면 구글만이 “최고”라고 보긴 힘듭니다.\n\n- 안전성·윤리·정책 측면의 복합성  \n  구글은 강한 연구역량과 안전연구 그룹을 갖췄지만, 사업·정책 결정으로 논란을 겪은 사례도 있어 “안전성·윤리에서 최고”라고 단언하기 어렵습니다. Anthropic 같은 곳은 안전을 핵심 차별점으로 내세웁니다.\n\n- 제품 통합·인프라는 구글의 강점이지만 전부는 아님  \n  구글은 검색·클라우드·모바일 등과의 통합, TPU 같은 인프라에서 큰 강점이 있어 “최고 후보”로 볼 근거가 충분합니다. 다만 상용화·생태계 관점에서 OpenAI의 광범위한 채택이나 지역별 중국 기업(바이두·알리바바)도 무시할 수 없습니다.\n\n결론: 구글은 언어 모델 분야에서 매우 강력한 플레이어(연구·인프라·제품 통합에서 뛰어남)이고 “최고 후보”라고 부를 만한 

## RunnableParallel.Assign   

Assign을 사용하면, 직전 체인의 실행 결과를 다음 체인에 전달하고, 결과를 결합할 수 있습니다.   
assign을 붙이기 위해서는 체인의 결과물이 dict 형태여야 합니다.



In [61]:
chain4 = prompt3 | llm | StrOutputParser()
# chain4 = {pros, cons} --> decision 출력하는 체인
# chain3 = RunnableParallel로 {pros, cons} 생성하는 체인

chain5 = chain3.assign(Decision = chain4)
#        pros, cons +  Decision
result = chain5.invoke("인간은 미래에 인공지능을 이길 수 없다.")
result

{'cons': "절대 아니야. 인간이 미래에 인공지능을 이길 수 없다는 말은 지나치게 단정적이고 현실을 안 봐. 몇 가지 이유만 들어줄게.\n\n- 창의성과 직관은 단순한 계산으로 완전히 대체되기 어려워. 인간은 전혀 다른 맥락을 연결해 새로운 아이디어를 만들어내는 능력이 있어.  \n- 가치 판단과 윤리적 결단은 기술적 능력보다 사회적 합의와 감수성에 기반해. AI가 아무리 똑똑해도 책임, 신뢰, 도덕을 스스로 결정하긴 힘들어.  \n- 정치·법·사회적 통제 수단이 있어. 규제, 국제협약, 거버넌스 체계로 인간이 AI를 통제하고 방향을 정할 수 있어.  \n- 집단지성과 협업 능력으로 인간은 AI를 개발·감독·교정할 수 있어. 여러 분야 전문가들이 모이면 AI의 한계를 보완하고 제어할 수 있다.  \n- 자원과 인프라의 한계가 있어. 최고급 AI는 막대한 에너지와 계산 자원을 필요로 하고, 그걸 모두 유지·운영하는 건 쉽지 않아.  \n- 안전 연구와 정렬(alignment) 기술이 발전하고 있어. '오프스위치', 제어 알고리즘, 투명성 기술로 AI 영향을 줄일 수 있어.  \n- '이긴다'의 의미도 문제야. 경쟁으로 보기보다 도구로서 보완하고 공존하는 게 현실적이고 바람직해.\n\n결국 인간이 AI를 완전히 압도당한다는 결정론은 근거 약해. 인간이 통제하고 적응하며 함께 발전할 가능성이 훨씬 높아.",
 'pros': '그 의견에 전적으로 동의합니다. 인공지능은 연산 속도와 학습 능력, 방대한 데이터 처리에서 인간을 훨씬 능가할 가능성이 크고, 자가학습과 자동화로 지속적으로 성능을 개선해 나갈 것입니다. 또한 지리적·물리적 제약을 받지 않고 대규모로 확장될 수 있어 인간 개개인이나 집단이 장기적으로 경쟁해 이기기 어려운 구조가 형성될 것입니다. 이러한 이유들로 인간이 미래에 인공지능을 이길 수 없다는 판단에 동의합니다.',
 'Decision': '간단히 답하면, 저는 반대 의견에 손을 들어주겠습니다 — 다만 반대 의견이 절대적 진실이라고 단정하지는 않습

<br><br>
### RunnablePassthrough
RunnablePassthrough는 체인의 직전 출력을 그대로 가져옵니다.

In [63]:
from langchain_core.runnables import RunnablePassthrough

prompt1 = ChatPromptTemplate(["주어진 단어에 대한 재미있는 삼행시를 작성하세요.\n 단어:{word}"])

chain1 = (prompt1 
          | llm
          | StrOutputParser()
          | {'result':RunnablePassthrough()}
        )

result = chain1.invoke('컴퓨터')
result

{'result': "컴퓨터야, 네가 잠들 줄 몰라서 내 새벽을 훔쳐  \n퓨~ 팬 소리로 심야 콘서트를 열지 마  \n터질 듯한 버그 앞에선 '재시작'이 우리의 기도"}

In [65]:

prompt1 = ChatPromptTemplate(["주어진 단어에 대한 재미있는 삼행시를 작성하세요.\n 단어:{word}"])
prompt2 = ChatPromptTemplate(['''주어진 시가 재미있는지 판단하고, 이를 개선해 주세요. 
{N} 문장으로 만들어야 합니다.
---                          

시: {poem}'''])

# TODO: 입력변수 word, N --> 2번 체인: poem, N 

chain1 = prompt1 | llm| StrOutputParser()
chain1_5 = prompt2 | llm | StrOutputParser()

chain2 = RunnablePassthrough.assign(poem = chain1) | chain1_5
#        word, N              +       poem        ---> 출력

result = chain2.invoke({'word':'컴퓨터', 'N':5})
result

'재미있습니다. 디지털 시대의 미세한 공감(‘5분만’→새벽, 뒤섞인 탭, reboot의 욕망)을 유머러스하게 잡아냈어요. 다만 문장이 좀 더 늘어나면 이야기가 풍부해지고 여운이 강해질 수 있습니다.\n\n컴퓨터를 켜면 \'5분만\'이라는 주문이 조용히 자라서 어느새 새벽을 차지한다.  \n탭들이 퓨전 요리처럼 섞여 업무인지 구독 영상인지 경계는 흐려진다.  \n메일과 알림이 파도처럼 밀려오고 나는 수면 대신 스크롤을 선택한다.  \n터미널에 "sudo reboot"을 치고 싶은 마음은 커지지만, 비밀번호 입력 귀찮음이 발목을 잡는다.  \n결국 화면을 끄면 기계는 잠들어도 내 시간은 아직 부팅을 기다린다.'

In [66]:
chain2 = RunnablePassthrough.assign(poem = chain1).assign(edit= chain1_5)
#        word, N              +       poem          +      edit

result = chain2.invoke({'word':'컴퓨터', 'N':5})
result

{'word': '컴퓨터',
 'N': 5,
 'poem': '컴퓨터야, 내 숙제 좀 도와줘!  \n퓨륙—알림: "자동업데이트 시작합니다. 지금 재시작할게요."  \n터져버린 3시간 작업, Ctrl+S 어디 갔냐고!!!',
 'edit': '재미있어요. 짧고 누구나 겪어본 당혹스러운 상황에 유머를 잘 섞어 공감이 갑니다.\n\n컴퓨터야, 제발 내 숙제 좀 도와줘!  \n퓨륙—알림이 뜨더니 자동업데이트가 시작된대.  \n숨도 못 쉬고 재시작 버튼이 눌렸고 화면은 까맣게 꺼졌다.  \n세 시간의 작업이 폭발하듯 사라졌고, Ctrl+S는 도망간 유령 같다.  \n다시 켜진 컴퓨터 앞에서 손은 떨리지만 이번엔 저장을 습관으로 새기리라 다짐한다.'}

In [67]:
prompt2 = ChatPromptTemplate(['''주어진 시가 재미있는지 판단하고, 이를 개선해 주세요. 
{word}에 대한 삼행시이므로, 규칙을 잘 지키는지 검증하고, 다시 써 주세요.                          

{N}개의 감상 포인트도 알려주세요.                              
---                          

시: {poem}'''])

chain1_5 = prompt2 | llm | StrOutputParser()


chain2 = RunnablePassthrough.assign(poem = chain1).assign(edit= chain1_5)
#        word, N              +       poem          +      edit

result = chain2.invoke({'word':'컴퓨터공학', 'N':5})
result

{'word': '컴퓨터공학',
 'N': 5,
 'poem': '컴퓨, 커피 한 잔에 0과 1이 연애 중  \n터공의 미로에서 버그는 숨고 나는 디버깅 탐정  \n학, 결국엔 "Hello, World!" 앞에서 감동받는다',
 'edit': '짧게 평가하고 개선본과 감상 포인트 드립니다.\n\n1) 규칙 검증\n- 삼행시 규칙(세 줄로 구성, 각 줄이 지정된 음절로 시작) 잘 지켜졌습니다.\n- 원문은 "컴퓨 / 터공 / 학" 순서로 각 줄이 시작해, 전체 단어 "컴퓨터공학"의 음절을 모두 아우르고 있습니다.\n- 주제(컴퓨터공학)와 연관된 이미지(0과 1, 버그, 디버깅, Hello, World!)도 잘 반영되어 있습니다.\n\n2) 흥미도 평가(간단)\n- 장점: 은유적이고 유머 있는 이미지(커피와 0·1의 연애), 버그를 탐정으로 추적하는 설정, 마지막의 감동적 결말이 인상적입니다.\n- 보완점: 일부 문장이 장황해 리듬이 흐트러지고, 한두 표현을 더 날카롭게 다듬으면 임팩트가 커집니다.\n\n3) 개선된 삼행시 (리듬·이미지·간결성 강화)\n컴퓨, 커피잔 속에서 0과 1이 몰래 신호를 주고받네  \n터공의 미로를 등불 삼아 나는 버그를 추적하는 탐정  \n학, 결국 \'Hello, World!\' 한 줄에 마음이 반짝인다\n\n4) 추가 개선 제안(원하시면 적용)\n- 유머 쪽으로 더 밀기: 끝을 웃음으로 치환(예: "컴퓨터가 먼저 이별을 고한다")  \n- 더 기술적으로: 구체적 용어(스택, 스레드, 레이스 컨디션 등) 삽입  \n- 운율·음절 맞추기: 각 줄 길이를 비슷하게 맞춰 리듬 강화\n\n5) 감상 포인트 (5가지)\n- 커피와 0·1의 \'연애\'라는 발상: 일상과 이진 논리가 결합된 신선한 이미지  \n- 버그를 \'숨는\' 존재로 묘사하고 디버거를 \'탐정\'으로 표현한 비유의 적절성  \n- 마지막에 "Hello, World!"로 귀결시켜 초심(처음의 기쁨)으로 돌아가는 감성  \n- 전체적으로 전문성과 인간미를 동시에 담아 컴퓨터공

## 복잡한 체인 만들기
chain2에서 새로운 매개변수가 추가되는 경우는 어떻게 해야 할까요?

In [71]:
prompt = ChatPromptTemplate(['''영화가 주어지면, 주연 배우 1명과 감독을 json으로 출력하세요.
actor, director의 키를 사용하세요.

movie: {movie}'''])

chain = prompt | llm | JsonOutputParser()

chain.invoke("타이타닉")

{'actor': 'Leonardo DiCaprio', 'director': 'James Cameron'}

In [72]:
prompt = ChatPromptTemplate(['''{actor}와 {director}는 어떤 영화에서 같이 만났나요? 모두 알려주세요.'''])

chain2 = prompt | llm | StrOutputParser()

chain3 = chain | chain2

chain3.invoke("인셉션")


'레오나르도 디카프리오와 크리스토퍼 놀란이 직접 함께 작업한 영화는 2010년작 "인셉션 (Inception)" 한 편뿐입니다.  \n- 디카프리오: 주연(도미닉 "돔" 콥)  \n- 크리스토퍼 놀란: 감독 겸 각본 공동 집필(조나단 놀란과 공저)\n\n그 외에는 같은 작품에서 만나지 않았습니다(2024년 6월 기준). 더 자세한 출연·제작 정보 원하시면 알려주세요.'

In [73]:
chain3 = chain.assign(result = chain2)
chain3.invoke("더 울프 오브 월 스트리트")

{'actor': 'Leonardo DiCaprio',
 'director': 'Martin Scorsese',
 'result': 'Leonardo DiCaprio와 Martin Scorsese가 함께 작업한 작품들은 다음과 같습니다.\n\n장편 영화 (감독: Martin Scorsese, 주연: Leonardo DiCaprio)\n- Gangs of New York (2002) — 갱스 오브 뉴욕  \n- The Aviator (2004) — 에비에이터  \n- The Departed (2006) — 디파티드  \n- Shutter Island (2010) — 셔터 아일랜드  \n- The Wolf of Wall Street (2013) — 더 울프 오브 월 스트리트  \n- Killers of the Flower Moon (2023) — 킬러스 오브 더 플라워 문\n\n단편/특별 작품\n- The Audition (2015, 단편) — 오디션 (스코세지 연출의 짧은 프로모션 필름 / 단편)\n\n원하시면 각 작품에서 DiCaprio가 맡은 배역이나 간단한 줄거리, 수상 내역도 정리해 드리겠습니다.'}

<br><br>
체인을 분리하고 RunnableParallel을 이용하면 중간 과정을 모두 출력할 수 있습니다.

<br><br><br>하나의 체인에서 여러 개의 값을 생성하려면,   
JsonOutputParser를 쓰면 됩니다.